In [ ]:
!pip install facenet-pytorch --no-deps -q
!pip install timm -q
print("Dependências instaladas")

In [ ]:
import sys
import torch
import os
from pathlib import Path

print("=== Ambiente ===")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n=== Inputs ===")
for item in sorted(os.listdir('/kaggle/input/')):
    print(f"  /kaggle/input/{item}")

In [ ]:
import sys
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import sys
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from facenet_pytorch import MTCNN

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, roc_curve
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

POSSIBLE_PATHS = [
    '/kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos',
]
VIDEOS_PATH = None
for p in POSSIBLE_PATHS:
    if Path(p).exists():
        VIDEOS_PATH = Path(p)
        break
assert VIDEOS_PATH is not None, "DFDC não encontrado!"

WORK_DIR = Path('/kaggle/working')
FACES_DIR = WORK_DIR / 'faces'
FACES_DIR.mkdir(exist_ok=True, parents=True)

FRAMES_PER_VIDEO = 15
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_EPOCHS = 8
LEARNING_RATE = 3e-5
NUM_WORKERS = 2

print(" Imports feitos")
print(f"   Python: {sys.version.split()[0]}")
print(f"   PyTorch: {torch.__version__}")
print(f"   Device: {device}")
print(f"   VIDEOS_PATH: {VIDEOS_PATH}")
print(f"   SEED: {SEED}")

In [ ]:
POSSIBLE_PATHS = [
    '/kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos',
]
VIDEOS_PATH = None
for p in POSSIBLE_PATHS:
    if Path(p).exists():
        VIDEOS_PATH = Path(p)
        break

assert VIDEOS_PATH is not None, "Nenhum caminho do DFDC encontrado!"
print(f"DFDC em: {VIDEOS_PATH}")

WORK_DIR = Path('/kaggle/working')
FACES_DIR = WORK_DIR / 'faces'
FACES_DIR.mkdir(exist_ok=True, parents=True)

FRAMES_PER_VIDEO = 15
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_EPOCHS = 8
LEARNING_RATE = 3e-5
NUM_WORKERS = 2

print(f"   Configurações definidas")
print(f"   Frames/vídeo: {FRAMES_PER_VIDEO}")
print(f"   Tamanho imagem: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Épocas: {NUM_EPOCHS}")

In [ ]:
with open(VIDEOS_PATH / 'metadata.json', 'r') as f:
    metadata = json.load(f)

df = pd.DataFrame([
    {'filename': k, 'label': v['label'], 'original': v.get('original')}
    for k, v in metadata.items()
])

print(f"Total vídeos: {len(df)}")
print(f"\nDistribuição:")
print(df['label'].value_counts())
print(f"\nREAL únicos (identities): {df[df['label']=='REAL']['filename'].nunique()}")
print(f"FAKEs com original definido: {df[df['label']=='FAKE']['original'].notna().sum()}")

fakes_per_real = df[df['label']=='FAKE'].groupby('original').size()
print(f"\nFakes por cada REAL:")
print(f"  Média: {fakes_per_real.mean():.1f}")
print(f"  Min: {fakes_per_real.min()}, Max: {fakes_per_real.max()}")

In [ ]:
import random
import numpy as np
import torch

# Seed para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"SEED: {SEED}")

In [ ]:
POSSIBLE_PATHS = [
    '/kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos',
]
VIDEOS_PATH = None
for p in POSSIBLE_PATHS:
    if Path(p).exists():
        VIDEOS_PATH = Path(p)
        break
assert VIDEOS_PATH is not None, "Nenhum caminho do DFDC encontrado!"
print(f" DFDC em: {VIDEOS_PATH}")

WORK_DIR = Path('/kaggle/working')
FACES_DIR = WORK_DIR / 'faces'
FACES_DIR.mkdir(exist_ok=True, parents=True)

FRAMES_PER_VIDEO = 15
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_EPOCHS = 8
LEARNING_RATE = 3e-5
NUM_WORKERS = 2

print(f"   Configurações definidas")
print(f"   Frames/vídeo: {FRAMES_PER_VIDEO}")
print(f"   Tamanho imagem: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Épocas: {NUM_EPOCHS}")

In [ ]:
with open(VIDEOS_PATH / 'metadata.json', 'r') as f:
    metadata = json.load(f)

# DataFrame para análise
df = pd.DataFrame([
    {'filename': k, 'label': v['label'], 'original': v.get('original')}
    for k, v in metadata.items()
])

print(f"Total vídeos: {len(df)}")
print(f"\nDistribuição:")
print(df['label'].value_counts())
print(f"\nREAL únicos (identities): {df[df['label']=='REAL']['filename'].nunique()}")
print(f"FAKEs com original definido: {df[df['label']=='FAKE']['original'].notna().sum()}")

# Quantos fakes por cada real?
fakes_per_real = df[df['label']=='FAKE'].groupby('original').size()
print(f"\nFakes por cada REAL:")
print(f"  Média: {fakes_per_real.mean():.1f}")
print(f"  Min: {fakes_per_real.min()}, Max: {fakes_per_real.max()}")

In [ ]:
df['split'] = None

# 1. Split dos Reais 
real_identities = df[df['label']=='REAL']['filename'].tolist()
random.seed(SEED)
random.shuffle(real_identities)

n = len(real_identities)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_ids = set(real_identities[:n_train])
val_ids = set(real_identities[n_train:n_train+n_val])
test_ids = set(real_identities[n_train+n_val:])

# 2. Atribuir split aos REAIS
def split_real(filename):
    if filename in train_ids: return 'train'
    elif filename in val_ids: return 'val'
    elif filename in test_ids: return 'test'
    return 'unknown'

df.loc[df['label']=='REAL', 'split'] = df[df['label']=='REAL']['filename'].apply(split_real)

# 3. FAKEs
real_filenames = set(df[df['label']=='REAL']['filename'].tolist())
real_to_split = {
    **{r: 'train' for r in train_ids},
    **{r: 'val' for r in val_ids},
    **{r: 'test' for r in test_ids},
}

# 3a. FAKEs orphan, os originais não estão neste datasset de teste 
orphan_fakes = df[(df['label']=='FAKE') & 
                   (~df['original'].isin(real_filenames))]['filename'].tolist()
random.seed(SEED + 1)
random.shuffle(orphan_fakes)

n_orph = len(orphan_fakes)
n_orph_train = int(0.7 * n_orph)
n_orph_val = int(0.15 * n_orph)

orphan_split = {}
for i, f in enumerate(orphan_fakes):
    if i < n_orph_train:
        orphan_split[f] = 'train'
    elif i < n_orph_train + n_orph_val:
        orphan_split[f] = 'val'
    else:
        orphan_split[f] = 'test'

# 3b. Atribuir split aos FAKEs
def split_fake(row):
    if row['original'] in real_to_split:
        return real_to_split[row['original']]
    else:
        return orphan_split.get(row['filename'], 'unknown')

df.loc[df['label']=='FAKE', 'split'] = df[df['label']=='FAKE'].apply(split_fake, axis=1)

# === VERIFICAÇÃO ===
print("Distribuição final:")
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))

unknown_count = (df['split'] == 'unknown').sum()
print(f"\nUnknowns restantes: {unknown_count} (deve ser 0)")

print(f"\nResumo por split:")
for s in ['train', 'val', 'test']:
    n_real = ((df['split']==s) & (df['label']=='REAL')).sum()
    n_fake = ((df['split']==s) & (df['label']=='FAKE')).sum()
    n_total = n_real + n_fake
    print(f"  {s:5s}: {n_total:3d} vídeos ({n_real:3d} REAL + {n_fake:3d} FAKE) — ratio FAKE/total: {n_fake/n_total:.2%}")

In [ ]:
import os
for item in os.listdir('/kaggle/input/'):
    print(item)
for item in os.listdir('/kaggle/input/datasets/'):
    print(item)

In [ ]:
# ============================================================
# CARREGAR 6 DATASETS 
# face-swap (CelebDF + FF++ + DFDC) + GAN (140K + StyleGAN3) + reais (FFHQ)
# ============================================================
import pandas as pd
import random
from pathlib import Path

# CelebDF (face-swap em vídeos de celebridades)
CELEBDF_PATH = Path('/kaggle/input/datasets/amanrawat001/celeb-df-preprocessed/Celeb-DF Preprocessed')

celebdf_records = []
for split_folder in ['train', 'val', 'test']:
    for label_folder in ['real', 'fake']:
        folder = CELEBDF_PATH / split_folder / label_folder
        if not folder.exists():
            continue
        for img_path in folder.rglob('*'):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                celebdf_records.append({
                    'path': str(img_path),
                    'label': 'REAL' if label_folder == 'real' else 'FAKE',
                    'split': split_folder,
                    'source': 'celebdf'
                })

df_celebdf = pd.DataFrame(celebdf_records)
print(f"CelebDF faces carregadas: {len(df_celebdf)}")
print(df_celebdf.groupby(['split', 'label']).size().unstack(fill_value=0))

# FaceForensics++ (deepfakes clássicos em vídeo)
FF_PATH = Path('/kaggle/input/datasets/greatgamedota/faceforensics/cropped_images')

ff_records = []
for img_path in FF_PATH.rglob('*.png'):
    ff_records.append({
        'path': str(img_path),
        'label': 'FAKE',
        'source': 'ff++'
    })

ff_df = pd.DataFrame(ff_records)
ff_indices = list(range(len(ff_df)))
random.seed(SEED + 2)
random.shuffle(ff_indices)
n_ff = len(ff_df)
n_ff_train = int(0.7 * n_ff)
n_ff_val = int(0.15 * n_ff)
ff_df['split'] = 'test'
ff_df.iloc[ff_indices[:n_ff_train], ff_df.columns.get_loc('split')] = 'train'
ff_df.iloc[ff_indices[n_ff_train:n_ff_train+n_ff_val], ff_df.columns.get_loc('split')] = 'val'
print(f"FF++ faces carregadas: {len(ff_df)} (todas FAKE)")

# DFDC pré-extraído (face-swap variado)
FACES_PREEXTRACTED = Path('/kaggle/input/datasets/pedromorais05/dfdc-faces')

if FACES_PREEXTRACTED.exists():
    dfdc_records = []
    for label in ['REAL', 'FAKE']:
        folder = FACES_PREEXTRACTED / label
        if folder.exists():
            for img_path in folder.rglob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    dfdc_records.append({
                        'path': str(img_path),
                        'label': label,
                        'source': 'dfdc',
                        'source_video': img_path.name,
                    })
    
    df_dfdc = pd.DataFrame(dfdc_records)
    
    videos = df_dfdc['source_video'].str.extract(r'(.+?)_frame')[0].unique()
    random.seed(SEED)
    random.shuffle(videos)
    n = len(videos)
    train_vids = set(videos[:int(0.7*n)])
    val_vids   = set(videos[int(0.7*n):int(0.85*n)])
    
    def assign_split(sv):
        vid = sv.split('_frame')[0]
        if vid in train_vids: return 'train'
        elif vid in val_vids:  return 'val'
        else:                  return 'test'
    
    df_dfdc['split'] = df_dfdc['source_video'].apply(assign_split)
    df_dfdc = df_dfdc.drop(columns=['source_video'])
    print(f"DFDC faces carregadas: {len(df_dfdc)}")
    print(df_dfdc.groupby(['split', 'label']).size().unstack(fill_value=0))
else:
    df_dfdc = pd.DataFrame()
    print("DFDC pré-extraído não encontrado — a saltar")

# 140K Real and Fake Faces (StyleGAN2)
FACES_140K = Path('/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake')

records_140k = []
split_map_140k = {'train': 'train', 'valid': 'val', 'test': 'test'}

for split_folder in ['train', 'valid', 'test']:
    for label_folder in ['real', 'fake']:
        folder = FACES_140K / split_folder / label_folder
        if not folder.exists():
            continue
        all_imgs = list(folder.glob('*.jpg'))
        random.seed(SEED + 30)
        selected = random.sample(all_imgs, min(10000, len(all_imgs)))
        for img_path in selected:
            records_140k.append({
                'path': str(img_path),
                'label': 'REAL' if label_folder == 'real' else 'FAKE',
                'split': split_map_140k[split_folder],
                'source': '140k'
            })

df_140k = pd.DataFrame(records_140k)
print(f"140K faces carregadas: {len(df_140k)}")
print(df_140k.groupby(['split', 'label']).size().unstack(fill_value=0))

# StyleGAN3 (faces sintéticas modernas)
STYLEGAN3_PATH = Path('/kaggle/input/datasets/troykueh/real-vs-fake-faces-stylegan3')

stylegan3_records = []
if STYLEGAN3_PATH.exists():
    for folder_name, label_normalized in [('Real faces', 'REAL'), ('Fake faces', 'FAKE')]:
        folder = STYLEGAN3_PATH / folder_name
        if folder.exists():
            for img_path in folder.rglob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    stylegan3_records.append({
                        'path': str(img_path),
                        'label': label_normalized,
                        'source': 'stylegan3'
                    })
    
    if stylegan3_records:
        random.seed(SEED + 40)
        random.shuffle(stylegan3_records)
        n = len(stylegan3_records)
        n_train = int(0.70 * n)
        n_val = int(0.15 * n)
        for i, r in enumerate(stylegan3_records):
            if i < n_train:           r['split'] = 'train'
            elif i < n_train + n_val: r['split'] = 'val'
            else:                     r['split'] = 'test'

df_stylegan3 = pd.DataFrame(stylegan3_records)
print(f"StyleGAN3 faces carregadas: {len(df_stylegan3)}")
if len(df_stylegan3) > 0:
    print(df_stylegan3.groupby(['split', 'label']).size().unstack(fill_value=0))

# FFHQ (faces REAIS, fotos do mundo real)
FFHQ_PATH = Path('/kaggle/input/datasets/greatgamedota/ffhq-face-data-set/thumbnails128x128')

all_ffhq_imgs = list(FFHQ_PATH.rglob('*.png'))
print(f"FFHQ imagens encontradas: {len(all_ffhq_imgs)}")

random.seed(SEED + 20)
ffhq_selected = random.sample(all_ffhq_imgs, min(20000, len(all_ffhq_imgs)))

n_ffhq = len(ffhq_selected)
n_ffhq_train = int(0.70 * n_ffhq)
n_ffhq_val   = int(0.15 * n_ffhq)

ffhq_records = []
for i, img_path in enumerate(ffhq_selected):
    if i < n_ffhq_train:
        split_ffhq = 'train'
    elif i < n_ffhq_train + n_ffhq_val:
        split_ffhq = 'val'
    else:
        split_ffhq = 'test'
    ffhq_records.append({
        'path': str(img_path),
        'label': 'REAL',
        'split': split_ffhq,
        'source': 'ffhq'
    })

df_ffhq = pd.DataFrame(ffhq_records)
print(f"FFHQ faces selecionadas: {len(df_ffhq)} (todas REAL)")

# Juntar tudo
dfs_to_concat = [df_celebdf, ff_df, df_140k, df_stylegan3, df_ffhq]
if len(df_dfdc) > 0:
    dfs_to_concat.insert(2, df_dfdc)

combined_df = pd.concat(dfs_to_concat, ignore_index=True)

print(f"\n{'='*50}")
print(f"DATASET COMBINADO FINAL (6 datasets focados)")
print(f"{'='*50}")
print(f"Total: {len(combined_df)} faces")
print(f"  CelebDF:    {(combined_df['source']=='celebdf').sum()}")
print(f"  FF++:       {(combined_df['source']=='ff++').sum()}")
print(f"  DFDC:       {(combined_df['source']=='dfdc').sum()}")
print(f"  140K:       {(combined_df['source']=='140k').sum()}")
print(f"  StyleGAN3:  {(combined_df['source']=='stylegan3').sum()}")
print(f"  FFHQ:       {(combined_df['source']=='ffhq').sum()}")
print(f"\nPor split e label:")
print(combined_df.groupby(['split', 'label']).size().unstack(fill_value=0))
print(f"\nRatio REAL: {(combined_df['label']=='REAL').mean()*100:.1f}%")
print(f"Ratio FAKE: {(combined_df['label']=='FAKE').mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))

# Usar combined_df 
real_sample = combined_df[combined_df['label']=='REAL'].sample(6, random_state=SEED)
fake_sample = combined_df[combined_df['label']=='FAKE'].sample(6, random_state=SEED)

for i, (_, row) in enumerate(real_sample.iterrows()):
    img = Image.open(row['path'])
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"REAL ({row['source']})", color='green', fontsize=10)
    axes[0, i].axis('off')

for i, (_, row) in enumerate(fake_sample.iterrows()):
    img = Image.open(row['path'])
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"FAKE ({row['source']})", color='red', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.suptitle('Faces (REAL vs FAKE) — 3 datasets', y=1.02, fontsize=14)
plt.savefig(WORK_DIR / 'faces_sample.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = {'REAL': 0, 'FAKE': 1}
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        label = self.label_map[row['label']]
        
        if self.transform:
            img = self.transform(img)
        
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(degrees=15),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
    ], p=0.3),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.2),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_df = combined_df[combined_df['split']=='train']
val_df = combined_df[combined_df['split']=='val']
test_df = combined_df[combined_df['split']=='test']

train_ds = DeepfakeDataset(train_df, transform=train_transform)
val_ds = DeepfakeDataset(val_df, transform=eval_transform)
test_ds = DeepfakeDataset(test_df, transform=eval_transform)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
from torch.utils.data import WeightedRandomSampler

train_labels = [0 if l == 'REAL' else 1 for l in train_df['label']]
class_counts = Counter(train_labels)
class_weights = {cls: 1.0/count for cls, count in class_counts.items()}
sample_weights = [class_weights[l] for l in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

sample_batch, sample_labels = next(iter(train_loader))
print(f"Batch shape: {sample_batch.shape}")
print(f"Labels no batch: {Counter(sample_labels.numpy())}")
print(f"(Deve estar razoavelmente balanceado graças ao sampler)")

In [ ]:
def create_model(num_classes=2):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),            
        nn.Linear(in_features, num_classes)
    )
    return model

model = create_model(num_classes=2).to(device)

# Usar múltiplas GPUs se disponíveis 
if torch.cuda.device_count() > 1:
    print(f"A usar {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros totais: {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"\n Modelo pronto: ResNet-50 fine-tuned")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * imgs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_labels = []
    all_preds = []
    all_probs = []
    
    for imgs, labels in tqdm(loader, desc='Eval', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        
        total_loss += loss.item() * imgs.size(0)
        probs = torch.softmax(outputs, dim=1)[:, 1]  # prob de ser FAKE
        _, predicted = outputs.max(1)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    f1 = f1_score(all_labels, all_preds)
    
    return {
        'loss': avg_loss, 'acc': acc, 'auc': auc, 'f1': f1,
        'labels': np.array(all_labels),
        'preds': np.array(all_preds),
        'probs': np.array(all_probs),
    }

print(" Funções de treino/eval definidas")

In [ ]:
history = {'train_loss': [], 'train_acc': [],
           'val_loss': [], 'val_acc': [], 'val_auc': [], 'val_f1': []}

best_val_auc = 0
best_model_path = WORK_DIR / 'best_model.pth'
patience = 4
epochs_no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n━━━ Época {epoch}/{NUM_EPOCHS} ━━━")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['acc'])
    history['val_auc'].append(val_metrics['auc'])
    history['val_f1'].append(val_metrics['f1'])
    
    print(f"  Train: loss={train_loss:.4f}, acc={train_acc:.4f}")
    print(f"  Val:   loss={val_metrics['loss']:.4f}, acc={val_metrics['acc']:.4f}, "
          f"AUC={val_metrics['auc']:.4f}, F1={val_metrics['f1']:.4f}")
    
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        # Guardar correctamente, com ou sem DataParallel
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), best_model_path)
        else:
            torch.save(model.state_dict(), best_model_path)
        print(f"  Novo melhor modelo (AUC={best_val_auc:.4f})")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  Sem melhoria ({epochs_no_improve}/{patience})")
        if epochs_no_improve >= patience:
            print(f"  Early stopping na época {epoch}")
            break

print(f"\n Treino completo! Melhor AUC val: {best_val_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['train_loss'], label='Train', marker='o')
axes[0].plot(history['val_loss'], label='Val', marker='s')
axes[0].set_title('Loss por época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', marker='o')
axes[1].plot(history['val_acc'], label='Val', marker='s')
axes[1].set_title('Accuracy por época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['val_auc'], label='AUC', marker='o', color='purple')
axes[2].plot(history['val_f1'], label='F1', marker='s', color='orange')
axes[2].set_title('Métricas no val set')
axes[2].set_xlabel('Época')
axes[2].set_ylabel('Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Carregar o melhor modelo 
state_dict = torch.load(best_model_path)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_metrics = evaluate(model, test_loader, criterion, device)
print("=" * 50)
print("RESULTADOS FINAIS (Test set)")
print("=" * 50)
print(f"Accuracy: {test_metrics['acc']:.4f}")
print(f"AUC-ROC:  {test_metrics['auc']:.4f}")
print(f"F1-Score: {test_metrics['f1']:.4f}")
print(f"Loss:     {test_metrics['loss']:.4f}")

print("\n" + classification_report(
    test_metrics['labels'],
    test_metrics['preds'],
    target_names=['REAL', 'FAKE']
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusão
cm = confusion_matrix(test_metrics['labels'], test_metrics['preds'])
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['REAL', 'FAKE'],
    yticklabels=['REAL', 'FAKE'],
    ax=axes[0], cbar=False
)
axes[0].set_title('Matriz de Confusão (Test)')
axes[0].set_ylabel('Verdadeiro')
axes[0].set_xlabel('Previsto')

# Curva ROC
fpr, tpr, _ = roc_curve(test_metrics['labels'], test_metrics['probs'])
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['auc']:.4f}", linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[1].set_title('Curva ROC (Test)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / 'confusion_and_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Encontrar erros
errors_mask = test_metrics['preds'] != test_metrics['labels']

# Resetar índices ANTES do mask para garantir consistência
test_df_reset = test_df.reset_index(drop=True)
errors_df = test_df_reset[errors_mask].copy().reset_index(drop=True)
errors_df['predicted'] = test_metrics['preds'][errors_mask]
errors_df['confidence'] = test_metrics['probs'][errors_mask]

print(f"Total erros: {len(errors_df)} de {len(test_df_reset)} ({100*len(errors_df)/len(test_df_reset):.1f}%)")
print(f"  FAKE classificados como REAL: {(errors_df['label']=='FAKE').sum()}")
print(f"  REAL classificados como FAKE: {(errors_df['label']=='REAL').sum()}")

if len(errors_df) > 0:
    n_show = min(8, len(errors_df))
    
    errors_df['conf_distance'] = (errors_df['confidence'] - 0.5).abs()
    errors_df_sorted = errors_df.sort_values('conf_distance', ascending=False).head(n_show).reset_index(drop=True)
    
    rows = (n_show + 3) // 4
    fig, axes = plt.subplots(rows, 4, figsize=(16, 4*rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    
    for i in range(n_show):
        row = errors_df_sorted.iloc[i]
        img = Image.open(row['path'])
        axes[i].imshow(img)
        true_label = row['label']
        pred_label = 'FAKE' if row['predicted']==1 else 'REAL'
        conf = row['confidence']
        axes[i].set_title(f"Real: {true_label}\nPrev: {pred_label} ({conf:.2f})",
                         color='red', fontsize=10)
        axes[i].axis('off')
    
    # Esconder eixos extras se houver
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle('Amostras mal classificadas (modelo confiante mas errado)', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'error_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n Imagem guardada em {WORK_DIR / 'error_samples.png'}")
else:
    print("Sem erros — nada para visualizar.")

In [ ]:
import shutil
import json

# Resumo final em JSON
results = {
    'model': 'ResNet-50 fine-tuned',
    'num_epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'datasets': ['CelebDF', '140K Real/Fake', 'FFHQ'],
    'n_train_faces': len(train_df),
    'n_val_faces': len(val_df),
    'n_test_faces': len(test_df),
    'best_val_auc': best_val_auc,
    'test_accuracy': float(test_metrics['acc']),
    'test_auc': float(test_metrics['auc']),
    'test_f1': float(test_metrics['f1']),
    'test_loss': float(test_metrics['loss']),
}

with open(WORK_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(" Ficheiros guardados em /kaggle/working/:")
for item in WORK_DIR.iterdir():
    if item.is_file():
        size_mb = item.stat().st_size / 1e6
        print(f"   {item.name} ({size_mb:.1f} MB)")